In [29]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import nltk
import re
import string

from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [31]:
df = pd.read_csv('Phishing_Email.csv')

In [33]:
df.head()

,Unnamed: 0,Email Text,Email Type
0,0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email
1,1,the other side of * galicismos * * galicismo *...,Safe Email
2,2,re : equistar deal tickets are you still avail...,Safe Email
3,3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email
4,4,software at incredibly low prices ( 86 % lower...,Phishing Email


In [35]:
df.columns

Index(['Unnamed: 0', 'Email Text', 'Email Type'], dtype='object')

In [37]:
df.isnull().sum()

Unnamed: 0     0
Email Text    16
Email Type     0
dtype: int64

In [39]:
df.shape

(18650, 3)

In [41]:
df['Email Type'].value_counts()

Email Type
Safe Email        11322
Phishing Email     7328
Name: count, dtype: int64

In [43]:
df = df.rename(columns={
    'Email Text':'text',
    'Email Type':'label'
})

## Convert Labels to Numeric

In [45]:
df['label'] = df['label'].map({
    'Safe Email':0,
    'Phishing Email':1
})

In [48]:
df = df.dropna()

## Text Preprocessing

In [50]:
ps = PorterStemmer()

def transform_text(text):
    
    text = text.lower()
    
    text = re.sub(r'http\S+', '', text)
    
    text = re.sub(r'\d+', '', text)
    
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    words = text.split()
    
    words = [word for word in words if word not in stopwords.words('english')]
    
    words = [ps.stem(word) for word in words]
    
    return " ".join(words)

## Apply Preprocessing

In [52]:
df['transformed_text'] = df['text'].apply(transform_text)

## Feature Extraction

In [59]:
tfidf = TfidfVectorizer(max_features= 5000)
X = tfidf.fit_transform(df['transformed_text']).toarray()
y = df['label'] 

In [95]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.22, random_state=42)

## Train Naive Bayes

In [99]:
nb_model =MultinomialNB()
nb_model.fit(X_train, y_train)

MultinomialNB()

In [101]:
y_pred_nb = nb_model.predict(X_test)

In [103]:
y_pred_nb

array([0, 1, 0, ..., 1, 0, 0], dtype=int64)

## Check Accuracy

In [107]:
print(f" Accuracy : {accuracy_score(y_test, y_pred_nb)}")

 Accuracy : 0.9548780487804878


## Classification Report

In [110]:
print(classification_report(y_test, y_pred_nb))

              precision    recall  f1-score   support

           0       0.97      0.96      0.96      2446
           1       0.94      0.95      0.94      1654

    accuracy                           0.95      4100
   macro avg       0.95      0.95      0.95      4100
weighted avg       0.95      0.95      0.95      4100



## Train Logistic Regression

In [114]:
lr_model = LogisticRegression()
lr_model.fit(X_train, y_train)

LogisticRegression()

In [116]:
y_pred_lr = lr_model.predict(X_test)

In [120]:
y_pred_lr

array([0, 1, 0, ..., 1, 0, 0], dtype=int64)

In [123]:
print(f" Accuracy_lr : {accuracy_score(y_test, y_pred_lr)}")

 Accuracy_lr : 0.9612195121951219


## Save model with pickle

In [127]:
import pickle

In [139]:
pickle.dump(tfidf, open('models/vectorizer.pkl', 'wb'))

In [141]:
pickle.dump(nb_model, open('models/model.pkl', 'wb'))

## Test Loading

In [149]:
model = pickle.load(open('models/model.pkl', 'rb'))
vectorizer = pickle.load(open('models/vectorizer.pkl', 'rb'))

## Output:
* 1 → Fraud / Phishing
* 0 → Safe

## Test Prediction

In [153]:
sample = ["Your bank account has been suspended. Click here to verify immediately"]
sample_transformed = vectorizer.transform(sample)
prediction = model.predict(sample_transformed)
print(prediction)

[1]
